# Prerequirements & Setup
Start by making sure you have the required files in the proper structure before trying this, because it requires you to mount the drive in order to access the python scripts and datasets. The files should be accessible via our repo.

## Installing Dependencies
First, we need to install the required Python packages that are not pre-installed in Google Colab. This includes `transformers` for the language models, `datasets` for loading data, and `openai` (if your scripts interact with OpenAI's API).

In [ ]:
# Install necessary dependencies not pre-installed in Colab
!pip install transformers datasets openai

In [ ]:
# Consolidated imports from run.py, custom_datasets.py, and calculate_metrics.py
import argparse
import datetime
import functools
import json
import os
import random
import re
import time
from multiprocessing.pool import ThreadPool

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import tqdm
import transformers

from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve
)

In [ ]:
# Check if PyTorch is using the GPU version and if a GPU is available
print("PyTorch version:", torch.__version__)
print("Is CUDA (GPU) available?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
Is CUDA (GPU) available? True
GPU Device Name: NVIDIA RTX PRO 6000 Blackwell Server Edition


## Mounting Google Drive
Since the project files (like `run.py`) and datasets are stored in your Google Drive, we need to mount it to the Colab environment. When you run this, Colab will prompt you to grant access to your Drive.

**Note:** In the code cell after this one, you MUST update the `PROJECT_PATH` variable to point to the exact folder where you uploaded the DetectGPT files.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

# Define the path to your project folder. Where ever you set your project folder to be, set it here
# Example:
# PROJECT_PATH = "/content/drive/MyDrive/{YOUR FOLDER LOCATION}"
PROJECT_PATH = "/content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT"


# Change the current working directory to your project folder
os.chdir(PROJECT_PATH)
print("Current Working Directory:", os.getcwd())

# Set up a cache directory in your drive for HuggingFace models
cache_dir = os.path.join(PROJECT_PATH, "hf_cache")
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir

print("HuggingFace cache set to:", os.environ['HF_HOME'])

Current Working Directory: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT
HuggingFace cache set to: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/hf_cache


# Running DetectGPT

## Run Flags Reference

| Flag | Required | Example | Meaning |
|---|---|---|---|
| --output_name | Recommended | smoke_hc3 | Name of the run output folder so results are easy to identify. |
| --dataset | Yes | hc3_all | Dataset key to load and evaluate. |
| --base_model_name | Yes | gpt2-medium | Base causal language model used for generation and scoring. |
| --mask_filling_model_name | Yes (for perturbation runs) | t5-base | Mask infilling model used to create perturbed text. |
| --n_perturbation_list | Yes (for perturbation runs) | 3 or 1,3,10 | Number of perturbations per sample. You can pass a comma-separated list. |
| --n_samples | Yes | 50 | Number of examples to evaluate. Smaller is faster. |
| --pct_words_masked | Recommended | 0.3 | Approximate fraction of words to mask before refill perturbation. |
| --skip_baselines | Optional switch | include flag only | If present, skips baseline metrics and runs perturbation experiments only. |
| --cache_dir | Recommended | hf_cache (if running in colab) or C:/hf_home (if running on local) | Directory for model/dataset cache to avoid repeated downloads. |

## Other Useful optional flags:

| Flag | Needed | Example value | What it means |
|---|---|---|---|
| --batch_size | Optional | 10 | Batch size for scoring/supervised loops. Lower if memory is tight. |
| --chunk_size | Optional | 10 | Chunk size for perturbation generation. Lower if VRAM/RAM is limited. |
| --span_length | Optional | 2 | Number of contiguous tokens per masked span. |
| --n_perturbation_rounds | Optional | 1 | How many times perturbation is recursively applied. |

## Small Smoke Test Command
```shell
python run.py --output_name smoke_hc3 --dataset hc3_all --base_model_name gpt2 --mask_filling_model_name t5-small --n_perturbation_list 3 --n_samples 50 --pct_words_masked 0.3 --skip_baselines --cache_dir C:/hf_home --batch_size 10 --chunk_size 10
```
## Reusable Template
```shell
python run.py --output_name YOUR_RUN_NAME --dataset YOUR_DATASET --base_model_name gpt2-medium --mask_filling_model_name t5-base --n_perturbation_list 3 --n_samples 500 --pct_words_masked 0.3 --skip_baselines --cache_dir hf_cache
```





## Smoke Test 1: Testing using the original dataset (Writing Prompt)
This is a small "smoke test" designed to run quickly and verify that your environment is set up correctly without errors. It processes only 50 samples from the `writing` dataset.

Flags used in this run:

* `--output_name tiny_50`
* `--dataset writing`
* `--base_model_name gpt2`
* `--mask_filling_model_name t5-small`
* `--n_perturbation_list 5`
* `--n_samples 50`
* `--pct_words_masked 0.3`
* `--span_length 2`
* `--batch_size 4`
* `--chunk_size 4`
* `--skip_baselines`
* `--cache_dir hf_cache`

In [ ]:
# !python run.py --output_name tiny_50 --dataset writing --base_model_name gpt2 --mask_filling_model_name t5-small --n_perturbation_list 5 --n_samples 50 --skip_baselines --cache_dir hf_cache

## Smoke Test 2: Testing using HC3 Unified dataset
This test ensures that the pipeline works with a different dataset (`hc3_all`). It evaluates another 50 samples using similar settings to the first smoke test.

In [ ]:
# !python run.py --dataset hc3_all --base_model_name gpt2 --mask_filling_model_name t5-small --n_samples 50 --n_perturbation_list 3 --skip_baselines --cache_dir hf_cache --output_name smoke_hc3

## Medium Test (using HC3 Unified)
Once you confirm the smoke tests work, you can run a larger experiment.

**Note:** The command in the next cell is commented out (starts with `#`). To run it, remove the `#` symbol.

In [ ]:
# !python run.py --output_name main --dataset hc3_all --base_model_name gpt2-medium --mask_filling_model_name t5-base --n_perturbation_list 10 --n_samples 500 --cache_dir hf_cache

## Calculating Metrics
After generating the perturbations and scoring them in the steps above, we need to calculate the final evaluation metrics (like ROC AUC).

To calculate the metrics, you will need the outputted json file from running DetectGPT. They will usually look like this:

```perturbation_3_z_results.json```

⚠️ **IMPORTANT FOR NEW USERS:** The `--path` argument in the cell below currently points to a specific result file. **You must update this path** to match the actual `.json` file generated in your `results` folder from the runs you just completed above. This command assumes that the results folder is in the project working directory.

In [ ]:
# This is for Smoke Test 2. I just copied the output to find the directory and inputted here.
# !python calculate_metrics.py --path results/smoke_hc3/gpt2-t5-small-temp/2026-04-13-06-58-35-361222-fp32-0.3-1-hc3_all-50/perturbation_3_z_results.json

# Medium Tests

## Writing + GPT 2 + 64 Tokens

In [ ]:
!python run.py --prompt_tokens 64 --output_name SmokeTestWritingGpt2_2 --dataset writing --base_model_name openai-community/gpt2-large --mask_filling_model_name t5-base --n_perturbation_list 20 --n_samples 1000 --pct_words_masked 0.3 --cache_dir hf_cache --skip_baselines

In [ ]:
!python calculate_metrics.py --path results/SmokeTestWritingGpt2_2/openai-community_gpt2-large-t5-base-temp/2026-04-20-06-39-51-891591-fp32-0.3-1-writing-1000/perturbation_20_z_results.json

Selected threshold for target_fpr(1.00%): 2.9667 (FPR=0.0100, TPR=0.1330)
Selected threshold for target_fpr(0.10%): 3.6442 (FPR=0.0010, TPR=0.0270)
{
  "experiment_name": "perturbation_20_z_results.json",
  "detection_method": "DetectGPT",
  "model_used": "openai-community/gpt2-large",
  "dataset_used": "writing",
  "num_samples": 2000,
  "additional_details": {
    "additional_models_used": [
      "t5-base"
    ],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.2327209098862642,
    "accuracy": 0.5615,
    "precision": 0.9300699300699301,
    "recall": 0.133,
    "tpr": 0.133,
    "auc_roc": 0.813719
  },
  "metrics_at_0.1pct_fpr": {
    "f1": 0.05252918287937743,
    "accuracy": 0.513,
    "precision": 0.9642857142857143,
    "recall": 0.027,
    "tpr": 0.027,
    "auc_roc": 0.813719
  }
}


# Big Test: HC3 + GPT2-Large + 5000

In [ ]:
!python run.py --output_name Final_GPT2 --dataset hc3_all_10000 --base_model_name openai-community/gpt2-large --mask_filling_model_name t5-large --n_perturbation_list 20 --n_samples 5000 --cache_dir hf_cache --skip_baselines

Using device: cuda
Saving results to absolute path: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/tmp_results/Final_GPT2/openai-community_gpt2-large-t5-large-temp/2026-04-23-07-46-33-262220-fp32-0.3-1-hc3_all_10000-5000
Using cache dir hf_cache
Using prompt_tokens=30 for non-pubmed sample generation
Loading BASE model openai-community/gpt2-large...
model.safetensors: 100% 3.25G/3.25G [01:21<00:00, 39.7MB/s]
Loading weights: 100% 436/436 [00:00<00:00, 1007.19it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading mask filling model t5-large...
Loading weights: 100% 509/509 [00:00<00:00, 936.47it/s, Materializing param=shared.weight]
MOVING BASE MO

In [12]:
!python calculate_metrics.py --path results/Final_GPT2/openai-community_gpt2-large-t5-large-temp/2026-04-23-07-46-33-262220-fp32-0.3-1-hc3_all_10000-5000/perturbation_20_z_results.json

Selected threshold for target_fpr(1.00%): 1.8765 (FPR=0.0097, TPR=0.3349)
Selected threshold for target_fpr(0.10%): 2.6757 (FPR=0.0006, TPR=0.0331)
{
  "experiment_name": "perturbation_20_z_results.json",
  "detection_method": "DetectGPT",
  "model_used": "openai-community/gpt2-large",
  "dataset_used": "hc3_all_10000",
  "num_samples": 3500,
  "additional_details": {
    "additional_models_used": [
      "t5-large"
    ],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.4980875478113047,
    "accuracy": 0.6625714285714286,
    "precision": 0.9718076285240465,
    "recall": 0.33485714285714285,
    "tpr": 0.33485714285714285,
    "auc_roc": 0.9502125714285714
  },
  "metrics_at_0.1pct_fpr": {
    "f1": 0.06412382531785517,
    "accuracy": 0.5162857142857142,
    "precision": 0.9830508474576272,
    "recall": 0.03314285714285714,
    "tpr": 0.03314285714285714,
    "auc_roc": 0.9502125714285714
  }
}


# Big Test: HC3 + Falcon + 5000

In [ ]:
!python run.py --output_name Final_Falcon --prompt_tokens 64 --dataset hc3_all_10000 --base_model_name tiiuae/falcon-7b --mask_filling_model_name t5-large --n_perturbation_list 20 --n_samples 5000 --cache_dir hf_cache --skip_baselines

Using device: cuda
Saving results to absolute path: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/tmp_results/Final_Falcon/tiiuae_falcon-7b-t5-large-temp/2026-04-23-08-39-34-808691-fp32-0.3-1-hc3_all_10000-5000
Using cache dir hf_cache
Using prompt_tokens=64 for non-pubmed sample generation
Loading BASE model tiiuae/falcon-7b...
Loading weights: 100% 196/196 [01:14<00:00,  2.63it/s, Materializing param=transformer.word_embeddings.weight]
The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Loading mask filling model t5-large...
Loading weights: 100% 509/509 [00:00<00:00, 845.28it/s, Materializing param=shared.weight]
MOVING BASE MODEL TO GPU...DONE (69.24s)
Loading dataset hc3_all_10000...
Token indices sequence length is longer

In [11]:
!python calculate_metrics.py --path results/Final_Falcon/tiiuae_falcon-7b-t5-large-temp/2026-04-23-08-39-34-808691-fp32-0.3-1-hc3_all_10000-5000/perturbation_20_z_results.json

Selected threshold for target_fpr(1.00%): 3.1573 (FPR=0.0091, TPR=0.0440)
Selected threshold for target_fpr(0.10%): 3.8612 (FPR=0.0006, TPR=0.0074)
{
  "experiment_name": "perturbation_20_z_results.json",
  "detection_method": "DetectGPT",
  "model_used": "tiiuae/falcon-7b",
  "dataset_used": "hc3_all_10000",
  "num_samples": 3500,
  "additional_details": {
    "additional_models_used": [
      "t5-large"
    ],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.08355941399891481,
    "accuracy": 0.5174285714285715,
    "precision": 0.8279569892473119,
    "recall": 0.044,
    "tpr": 0.044,
    "auc_roc": 0.7228512653061224
  },
  "metrics_at_0.1pct_fpr": {
    "f1": 0.01473922902494331,
    "accuracy": 0.5034285714285714,
    "precision": 0.9285714285714286,
    "recall": 0.0074285714285714285,
    "tpr": 0.0074285714285714285,
    "auc_roc": 0.7228512653061224
  }
}


# Big Test: HC3 + Falcon Instruct + 5000

In [ ]:
!python run.py --output_name Final_FalInstruct --prompt_tokens 64 --dataset hc3_all_10000 --base_model_name tiiuae/falcon-7b-instruct --mask_filling_model_name t5-large --n_perturbation_list 20 --n_samples 5000 --cache_dir hf_cache --skip_baselines

In [ ]:
!python calculate_metrics.py --path results/

# Avoidance Techniques

## Recursive Paraphrasing

In [ ]:
!python avoidance_run.py --cache_dir hf_cache --skip_baselines --dataset avoidance_recursive_hc3 --n_samples 5000 --n_perturbation_list 20 --prompt_tokens 64 --mask_filling_model_name t5-large

In [ ]:
!python calculate_metrics.py --path results/gpt2-medium-t5-large-temp/2026-04-22-22-32-16-944177-fp32-0.3-1-avoidance_recursive_hc3-5000/perturbation_20_d_results.json

Selected threshold for target_fpr(1.00%): 0.2808 (FPR=0.0091, TPR=0.0080)
Selected threshold for target_fpr(0.10%): 0.7432 (FPR=0.0000, TPR=0.0010)
{
  "experiment_name": "perturbation_20_d_results.json",
  "detection_method": "DetectGPT",
  "model_used": "gpt2-medium",
  "dataset_used": "avoidance_recursive_hc3",
  "num_samples": 1988,
  "additional_details": {
    "additional_models_used": [
      "t5-large"
    ],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.01582591493570722,
    "accuracy": 0.4994969818913481,
    "precision": 0.47058823529411764,
    "recall": 0.008048289738430584,
    "tpr": 0.008048289738430584,
    "auc_roc": 0.5376565226368271
  },
  "metrics_at_0.1pct_fpr": {
    "f1": 0.0020100502512562816,
    "accuracy": 0.5005030181086519,
    "precision": 1.0,
    "recall": 0.001006036217303823,
    "tpr": 0.001006036217303823,
    "auc_roc": 0.5376565226368271
  }
}


## Stylistic Cleanup

In [ ]:
!python avoidance_run.py --cache_dir hf_cache --skip_baselines --dataset hc3_stylisticCleanup --n_samples 5000 --n_perturbation_list 20 --prompt_tokens 64 --pct_words_masked 0.2 --mask_filling_model_name t5-large

Using device: cuda
Saving results to absolute path: /content/drive/MyDrive/CSCI 544: Applied Natural Language Processing/Project/DetectGPT/tmp_results/gpt2-medium-t5-large-temp/2026-04-23-06-19-45-042224-fp32-0.2-1-hc3_stylisticCleanup-5000
Using cache dir hf_cache
Using prompt_tokens=64 for non-pubmed sample generation
Loading BASE model gpt2-medium...
Loading weights: 100% 292/292 [00:21<00:00, 13.74it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading mask filling model t5-large...
model.safetensors: 100% 2.95G/2.95G [00:46<00:00, 63.8MB/s]
Loading weights: 100% 509/509 [00:00<00:00, 881.37it/s, Materializing param=shared.weight]
MOVING BASE MODEL TO GPU...DONE (20.08s)
Loading dataset hc3_stylist

In [6]:
!python calculate_metrics.py --path results/gpt2-medium-t5-large-temp/2026-04-23-06-19-45-042224-fp32-0.2-1-hc3_stylisticCleanup-5000/perturbation_20_d_results.json

Selected threshold for target_fpr(1.00%): 0.1013 (FPR=0.0099, TPR=0.0284)
Selected threshold for target_fpr(0.10%): 0.1621 (FPR=0.0010, TPR=0.0059)
{
  "experiment_name": "perturbation_20_d_results.json",
  "detection_method": "DetectGPT",
  "model_used": "gpt2-medium",
  "dataset_used": "hc3_stylisticCleanup",
  "num_samples": 8090,
  "additional_details": {
    "additional_models_used": [
      "t5-large"
    ],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.05476190476190476,
    "accuracy": 0.5092707045735476,
    "precision": 0.7419354838709677,
    "recall": 0.02843016069221261,
    "tpr": 0.02843016069221261,
    "auc_roc": 0.7697597027262824
  },
  "metrics_at_0.1pct_fpr": {
    "f1": 0.011784925116621655,
    "accuracy": 0.5024721878862793,
    "precision": 0.8571428571428571,
    "recall": 0.005933250927070457,
    "tpr": 0.005933250927070457,
    "auc_roc": 0.7697597027262824
  }
}


## Perturbed Text

In [7]:
!python avoidance_run.py --cache_dir hf_cache --skip_baselines --dataset hc3_perturbed --n_samples 5000 --n_perturbation_list 20 --prompt_tokens 64 --pct_words_masked 0.2 --mask_filling_model_name t5-large

Streaming output truncated to the last 5000 lines.
Applying perturbations:  21% 1059/4985 [11:33<29:46,  2.20it/s]WARNING: 1 texts have no fills. Trying again [attempt 1].
Applying perturbations:  21% 1071/4985 [11:38<24:45,  2.63it/s]WARNING: 19 texts have no fills. Trying again [attempt 1].
Applying perturbations:  22% 1083/4985 [12:01<26:43,  2.43it/s]WARNING: 15 texts have no fills. Trying again [attempt 1].
Applying perturbations:  22% 1086/4985 [12:10<1:44:34,  1.61s/it]WARNING: 10 texts have no fills. Trying again [attempt 1].
Applying perturbations:  22% 1117/4985 [12:26<25:52,  2.49it/s]WARNING: 10 texts have no fills. Trying again [attempt 1].
Applying perturbations:  22% 1121/4985 [12:29<35:48,  1.80it/s]WARNING: 17 texts have no fills. Trying again [attempt 1].
Applying perturbations:  23% 1122/4985 [12:51<7:23:40,  6.89s/it]WARNING: 5 texts have no fills. Trying again [attempt 1].
Applying perturbations:  23% 1133/4985 [12:56<33:05,  1.94it/s]WARNING: 2 texts have no fills

In [10]:
!python calculate_metrics.py --path results/gpt2-medium-t5-large-temp/2026-04-23-17-00-01-128245-fp32-0.2-1-hc3_perturbed-5000/perturbation_20_d_results.json

Selected threshold for target_fpr(1.00%): 0.1375 (FPR=0.0098, TPR=0.0142)
Selected threshold for target_fpr(0.10%): 0.2647 (FPR=0.0008, TPR=0.0034)
{
  "experiment_name": "perturbation_20_d_results.json",
  "detection_method": "DetectGPT",
  "model_used": "gpt2-medium",
  "dataset_used": "hc3_perturbed",
  "num_samples": 9970,
  "additional_details": {
    "additional_models_used": [
      "t5-large"
    ],
    "notes": ""
  },
  "metrics_at_1pct_fpr": {
    "f1": 0.02781586679725759,
    "accuracy": 0.5022066198595787,
    "precision": 0.5916666666666667,
    "recall": 0.01424272818455366,
    "tpr": 0.01424272818455366,
    "auc_roc": 0.6619785937551873
  },
  "metrics_at_0.1pct_fpr": {
    "f1": 0.0067918497802636835,
    "accuracy": 0.5013039117352056,
    "precision": 0.8095238095238095,
    "recall": 0.0034102306920762286,
    "tpr": 0.0034102306920762286,
    "auc_roc": 0.6619785937551873
  }
}
